# RAG-Enhanced Coaching

This notebook extends the AI coaching system by enabling it to reference a curated collection of financial educational resources when generating coaching recommendations. By combining personalized user information with relevant financial knowledge, the system is able to provide more informative and actionable coaching specific to each user's financial circumstances.


## Initiate Environment


In [1]:
# Standard Library
import os, re, json, random
import joblib, pandas as pd, shap

# OpenAI - LLM
from dotenv import load_dotenv
from openai import OpenAI

# LangChain - Documents
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

# LangChain - Prompts & Parsing
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LangChain - Embeddings & Vector Store
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# LangChain - LLMs
from langchain_openai import ChatOpenAI

# Environment Variables & API Clients
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key = api_key)


## Knowledge Base Processing & Vector Store Management

The knowledge base pipeline converts financial coaching content into searchable vector embeddings for RAG, enabling cross-domain knowledge retrieval, metadata-driven management, and independent document updates.

### Purpose & Functionality:
* Knowledge Base = Vector Embeddings
* RAG Support for Personalized Coaching
* Document-to-Chunk Processing
* Metadata-Enriched Knowledge Retrieval

### Architecture:
* Unified Chroma Vector Store (`financial_coach`)
* Cross-Domain Semantic Search
* Metadata-Based Document Tracking
* Multi-Domain Knowledge Retrieval

### Document Management:
* Incremental Document Ingestion
* Collection Updates Without Rebuilds
* Metadata-Based Document Replacement
* Scalable Knowledge Base Expansion

### Benefits:
* Simplified Knowledge Base Maintenance
* Efficient Coaching Content Retrieval
* Document Traceability & Governance
* Scalable Foundation for Future Domains


In [2]:
# Create Vector Store
embeddings = OpenAIEmbeddings()

#vectorstore.delete_collection()
vectorstore = Chroma(
    collection_name = 'financial_coach',
    embedding_function = embeddings,
    persist_directory = 'chroma_db'
)


In [3]:
# Knowledge Base Processing Pipeline
def load_knowledge_document(file_path):

    # Load Document
    loader = TextLoader(file_path)
    documents = loader.load()

    text = documents[0].page_content

    # Domain
    domain_match = re.search(r'#\s+(.+)', text)
    domain = (
        domain_match.group(1).strip() if domain_match else 'Unknown'
    )

    # Tags
    tags_match = re.search(
        r'Tags:\s*(.*?)\n\nContent Type:', text, re.DOTALL
    )

    tags = []
    if tags_match:

        tags = [

            tag.strip()

            for tag in (
                tags_match.group(1).replace('\n', ' ').split(',')
            )

            if tag.strip()

        ]

    # Source File
    source = os.path.basename(file_path)

    # Split Into Sections
    sections = re.split(r'\n##\s+', text)[1:]

    # Build Chunks
    chunks = []
    for section in sections:

        section = section.strip()

        lines = section.split('\n')

        topic = lines[0].strip()

        content = '\n'.join(lines[1:]).strip()

        chunks.append(

            Document(

                page_content = content,

                metadata = {
                    'domain': domain,
                    'topic': topic,
                    'source': source,
                    'tags': tags
                }

            )

        )

    print(f'{source}: {len(chunks)} chunks created.')

    return chunks

# Add New Document
def add_knowledge_document(file_path, vectorstore):

    chunks = load_knowledge_document(file_path)
    vectorstore.add_documents(chunks)
    filename = os.path.basename(file_path)

    print(f'{filename}: {len(chunks)} chunks added.')
    print(f'Total Chunks: {vectorstore._collection.count()}\n')

# Replace Existing Document
def replace_knowledge_document(file_path, vectorstore):

    source = os.path.basename(file_path)

    # Remove Existing Chunks
    vectorstore._collection.delete(where = {'source': source})

    # Reload Updated Document
    add_knowledge_document(file_path, vectorstore)

    print(f'{source} replaced.\n')


In [4]:
# Add Documents
add_knowledge_document('knowledge_base/01_emergency_fund.txt', vectorstore)
add_knowledge_document('knowledge_base/02_debt_reduction.txt', vectorstore)


01_emergency_fund.txt: 10 chunks created.
01_emergency_fund.txt: 10 chunks added.
Total Chunks: 10

02_debt_reduction.txt: 14 chunks created.
02_debt_reduction.txt: 14 chunks added.
Total Chunks: 24



In [5]:
# Vector Store Validation
all_docs = vectorstore.get()
metadata_df = pd.DataFrame(all_docs['metadatas'])

display(metadata_df['source'].value_counts())


source
02_debt_reduction.txt    14
01_emergency_fund.txt    10
Name: count, dtype: int64

In [6]:
# Knowledge Base: Emergency Fund
all_docs = vectorstore.get()

count = 0
for doc, meta in zip(
    all_docs['documents'],
    all_docs['metadatas']
):

    if meta['source'] == '01_emergency_fund':

        count += 1

        if count == 1:  # Chunk Number

            print('Metadata:')
            print(meta)

            print('\nContent:')
            print(doc)

            break


## Financial Coaching

This section showcases the AI-powered coaching experience available within the web application. By combining user-specific financial data with retrieval-augmented generation, the system delivers personalized responses to coaching-related questions and provides context-aware financial guidance tailored to each user's situation.


### RAG-Enhanced Coaching

The financial coaching workflow combines Retrieval-Augmented Generation (RAG) with personalized financial analytics to deliver context-aware guidance. User questions are evaluated for relevance, supplemented with knowledge base content when available, and combined with the user's financial profile, including income, spending behavior, debt burden, emergency fund status, and financial goals, to generate personalized coaching recommendations grounded in both curated financial education content and objective financial data.

### Workflow:
* User financial question
* Personal finance relevance check
* Vector search for knowledge retrieval
* RAG for covered topics
* General LLM knowledge for uncovered topics
* Non-finance question filtering
* User profile enrichment
* Personalized coaching generation
* Evidence-grounded response delivery

### Benefits:
* Personalized financial guidance
* Knowledge-grounded responses
* Expandable coaching domains
* Reduced hallucination risk
* Personal finance topic boundaries


In [7]:
# Case Study: Test User
profile = {

    'persona': {
        'name': 'Near Successful',
        'cluster_id': 1,

        'description': (
            '''
                Users whose spending patterns closely resemble financially successful users but who generate 
                insufficient monthly surplus.
            '''
        )
    },

    'behavioral_alignment': {
        'success_similarity': 0.8259,
        'score': 82.6
    },

    'financial_capacity': {
        'monthly_income': 4811.00,
        'avg_monthly_spend': 4731.66,
        'monthly_surplus': 79.34,
        'surplus_ratio': 0.0165,
        'score': 1.6
    },

    'financial_stability': {
        'total_debt': 2262.00,
        'debt_to_income': 0.0392
    },

    'emergency_fund': {
        'balance': 0.00,
        'target': 14433.00,
        'progress': 0.0669
    },

    'forecasting': {
        'predicted_next_month_spending': 4942.56,
        'forecast_surplus': -131.56
    },

    'benchmark': {
        'monthly_savings_opportunity': 381.00
    },

    'spending_categories': [
        {
            'category': 'Shopping',
            'amount': 650.00
        },

        {
            'category': 'Transportation',
            'amount': 525.00
        },

        {
            'category': 'Food Discretionary',
            'amount': 480.00
        }
    ]
}


In [8]:
# Model
llm = ChatOpenAI(
    model = 'gpt-4o-mini', temperature = 0.5
)

# Topic Classification
def classify_question(question):

    prompt = ChatPromptTemplate.from_template(
        '''

            You are a financial topic classifier.

            Classify the user's question into ONE category:
            - emergency_fund
            - debt_reduction
            - other_finance
            - not_finance

            Return ONLY the category name.

            User Question: {question}

        '''
    )

    chain = prompt | llm | StrOutputParser()

    return chain.invoke({'question': question}).strip().lower()

# Retrieve Knowledge Base Context
def retrieve_context(question, vectorstore, k=4):

    results = vectorstore.similarity_search(question, k = k)

    context = '\n\n'.join([doc.page_content for doc in results])

    sources = list(set([
        doc.metadata.get('source', 'Unknown') for doc in results
    ]))

    return context, sources

# Profile Summary
def profile_summary(profile):

    return f'''

        Persona: {profile['persona']['name']}

        Monthly Income: ${profile['financial_capacity']['monthly_income']:,.0f}
        Monthly Spending: ${profile['financial_capacity']['avg_monthly_spend']:,.0f}
        Monthly Surplus: ${profile['financial_capacity']['monthly_surplus']:,.0f}

        Emergency Fund Balance: ${profile['emergency_fund']['balance']:,.0f}
        Emergency Fund Target: ${profile['emergency_fund']['target']:,.0f}

        Total Debt: ${profile['financial_stability']['total_debt']:,.0f}
        Debt-to-Income Ratio: {profile['financial_stability']['debt_to_income']:.1%}

        Forecast Surplus: ${profile['forecasting']['forecast_surplus']:,.0f}

        Potential Monthly Savings Opportunity:
        ${profile['benchmark']['monthly_savings_opportunity']:,.0f}

    '''

# Financial Coach
def ask_financial_coach(profile, question, vectorstore):

    category = classify_question(question)

    # Non-Finance Questions
    if category == 'not_finance':

        return ('I can only assist with personal finance related questions.')

    # Knowledge Base Topics
    if category in ['emergency_fund', 'debt_reduction']:

        context, sources = retrieve_context(question, vectorstore)

        source_type = 'Knowledge Base'

    # Other Finance Topics
    else:

        context = ''
        sources = []
        source_type = ('General Financial Knowledge')

    prompt = ChatPromptTemplate.from_template(
        '''

            You are a professional financial coach.

            User Financial Profile: {profile}

            Information Source: {source_type}

            Knowledge Base Context: {context}

            User Question: {question}

            Requirements:
            - Answer only personal finance questions.
            - If Knowledge Base Context is provided,
              rely primarily on that information.
            - Do not invent facts that contradict
              the knowledge base.
            - If no knowledge base context is provided,
              answer using general financial knowledge.
            - Personalize advice using the user's profile.
            - Focus on practical, actionable guidance.
            - Maintain a professional and supportive tone.
            - Limit responses to 250 words or less.

        '''
    )

    chain = (prompt | llm | StrOutputParser())

    response = chain.invoke({
        'profile': profile_summary(profile),
        'source_type': source_type,
        'context': context,
        'question': question
    })

    return response


In [9]:
# Example
response = ask_financial_coach(
    profile = profile,
    question = 'What is the best way to start an emergency fund?',
    vectorstore = vectorstore
)

print(response)


The best way to start an emergency fund is to set a specific, manageable savings goal and create a consistent plan to reach it. Given your current financial profile, here are some actionable steps:

1. **Set a Target**: Aim for a starter emergency fund of $500 to $1,000. This initial amount can cover many common emergencies and serves as a vital first milestone.

2. **Identify Savings Opportunities**: You have a potential monthly savings opportunity of $381. Consider allocating a portion of this toward your emergency fund. Even setting aside $100–$200 monthly can help you reach your goal quickly.

3. **Open a Dedicated Account**: Choose a high-yield savings account or a money market account to keep your emergency fund separate from your regular savings. This ensures easy access while earning some interest.

4. **Automate Savings**: If possible, set up an automatic transfer to your emergency fund account each month. This makes saving easier and lessens the temptation to spend that money

### Example Behaviors

The following examples illustrate how the LLM-based coaching framework handles different user questions. They demonstrate knowledge retrieval through RAG, use of general financial expertise when needed, and enforcement of personal finance topic boundaries.


In [10]:
# Topic: Emergency Fund
response = ask_financial_coach(
    profile, 'How much should I keep in my emergency fund?', vectorstore
)

print(response)


Given your financial profile, your current situation shows that you have no emergency fund, and your monthly spending is close to your income. It's crucial to establish a financial safety net to protect against unexpected expenses.

For your emergency fund, I recommend starting with a target of **$500 to $1,000**. This amount can cover many common emergencies, such as minor car repairs or medical expenses. Once you reach this starter emergency fund, you can aim to build it up to at least **one month of essential living expenses**, which would be around **$4,732** based on your current spending.

Since you have a potential monthly savings opportunity of **$381**, you can prioritize building your emergency fund. If you set aside **$250** each month, you can reach your starter emergency fund goal in just two months. After that, you can continue saving to build toward one month of expenses within the next few months.

Having this emergency fund will help prevent you from relying on debt du

In [11]:
# Topic: Debt Reduction
response = ask_financial_coach(
    profile, 'Should I use debt snowball or debt avalanche?', vectorstore
)

print(response)


Given your financial profile, choosing between the debt snowball and debt avalanche methods depends on your personal priorities and motivations.

**Debt Snowball Method:** This strategy focuses on paying off the smallest debts first. This approach can provide quick wins, boosting your motivation as you eliminate debts one by one. Since you have a total debt of $2,262, if you have smaller balances, you might find it encouraging to pay them off quickly.

**Debt Avalanche Method:** This approach prioritizes debts with the highest interest rates, which can save you more money in the long run. If any of your debts carry high interest, this method might be more financially efficient.

Given your current financial situation, where your monthly surplus is only $79 and you have a potential savings opportunity of $381, consider a hybrid approach. Start with the smallest balance to build momentum but also keep an eye on any higher interest debts. This way, you can make progress while staying moti

In [12]:
# Topic: Other Finance
response = ask_financial_coach(
    profile, 'How does a Roth IRA work?', vectorstore
)

print(response)


A Roth IRA (Individual Retirement Account) is a retirement savings account that allows you to contribute after-tax income, meaning you pay taxes on the money before you deposit it into the account. The key benefits of a Roth IRA include:

1. **Tax-Free Growth**: Your investments grow tax-free, and you won’t pay taxes on withdrawals in retirement, provided you meet certain conditions.

2. **Flexible Withdrawals**: You can withdraw your contributions (but not earnings) at any time without penalties, making it a flexible option for saving.

3. **No Required Minimum Distributions (RMDs)**: Unlike traditional IRAs, Roth IRAs do not require you to take distributions at a certain age, allowing your savings to grow longer.

For your financial situation, consider contributing to a Roth IRA once you establish your emergency fund and reduce your debt. With a monthly surplus of $79 and a potential savings opportunity of $381, you could prioritize building your emergency fund first to cover unexpec

In [13]:
# Topic: Not Finance
response = ask_financial_coach(
    profile, 'How do I make lasagna?', vectorstore
)

print(response)


I can only assist with personal finance related questions.
